In [ ]:
#Method 01: Hong's gabor Filtering
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.ndimage import sobel
from pathlib import Path
import os

# ============================================================
# 0. INPUT FOLDER (RGBA 4-channel images)
# ============================================================

INPUT_DIR = Path("enhanced_dataset_all/fg_final_rgba")   # <--- YOUR RGBA FOLDER
OUTPUT_DIR = Path("enhanced_dataset_all/fingerprint_gabor_batch")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Create subfolders
(OUTPUT_DIR / "normalized").mkdir(exist_ok=True)
(OUTPUT_DIR / "orientation").mkdir(exist_ok=True)
(OUTPUT_DIR / "frequency").mkdir(exist_ok=True)
(OUTPUT_DIR / "gabor_raw").mkdir(exist_ok=True)
(OUTPUT_DIR / "gabor_soft").mkdir(exist_ok=True)

# Parameters
BLOCK_SIZE      = 16
REL_THRESH      = 0.5
MIN_WAVELENGTH  = 4.0
MAX_WAVELENGTH  = 10.0

# ============================================================
# FUNCTIONS (your original ones, unchanged)
# ============================================================

def normalize_fingerprint(F, M0=0.0, V0=1.0):
    F = F.astype(np.float32)
    M = F.mean()
    V = F.var() + 1e-8
    N = np.zeros_like(F, dtype=np.float32)
    mask = F > M
    N[mask]  = M0 + np.sqrt(V0 * (F[mask] - M)**2 / V)
    N[~mask] = M0 - np.sqrt(V0 * (F[~mask] - M)**2 / V)
    return N

def estimate_orientation_and_reliability(N, block_size=16):
    rows, cols = N.shape
    Gx = sobel(N, axis=1, mode='reflect')
    Gy = sobel(N, axis=0, mode='reflect')

    orient = np.zeros_like(N, dtype=np.float32)
    rel    = np.zeros_like(N, dtype=np.float32)

    for y in range(0, rows - block_size + 1, block_size):
        for x in range(0, cols - block_size + 1, block_size):
            gx = Gx[y:y+block_size, x:x+block_size]
            gy = Gy[y:y+block_size, x:x+block_size]

            Vx = 2 * np.sum(gx * gy)
            Vy = np.sum(gx**2 - gy**2)
            theta = 0.5 * np.arctan2(Vx, Vy)

            J11 = np.sum(gx * gx)
            J22 = np.sum(gy * gy)
            J12 = np.sum(gx * gy)

            trace = J11 + J22
            det   = J11 * J22 - J12 * J12
            tmp   = np.sqrt(max(trace*trace / 4.0 - det, 0.0))

            lam1 = trace/2 + tmp
            lam2 = trace/2 - tmp

            if lam1 <= 0:
                r = 0
            else:
                r = 1.0 - lam2 / (lam1 + 1e-8)

            orient[y:y+block_size, x:x+block_size] = theta
            rel[y:y+block_size,   x:x+block_size] = np.clip(r, 0, 1)

    return orient, rel

def estimate_ridge_frequency(N, O, R, block_size=16,
                             rel_thresh=0.5,
                             min_wav=4.0,
                             max_wav=10.0):

    rows, cols = N.shape
    freq = np.zeros_like(N, dtype=np.float32)

    for y in range(0, rows - block_size + 1, block_size):
        for x in range(0, cols - block_size + 1, block_size):

            if R[y:y+block_size, x:x+block_size].mean() < rel_thresh:
                continue

            block = N[y:y+block_size, x:x+block_size]
            theta = O[y + block_size//2, x + block_size//2]

            angle_deg = (theta + np.pi/2) * 180/np.pi
            M = cv2.getRotationMatrix2D(
                (block_size/2 - 0.5, block_size/2 - 0.5),
                angle_deg, 1.0)

            rot = cv2.warpAffine(block, M, (block_size, block_size),
                                 flags=cv2.INTER_LINEAR,
                                 borderMode=cv2.BORDER_REFLECT_101)

            profile = np.mean(rot, axis=1)
            profile = (profile - profile.mean()) / (profile.std() + 1e-8)

            peaks, _ = find_peaks(profile, distance=int(min_wav))
            if len(peaks) < 2: continue

            d = np.diff(peaks).mean()
            if not (min_wav <= d <= max_wav): continue

            freq[y:y+block_size, x:x+block_size] = 1.0 / d

    return freq

def gabor_kernel(freq, theta, block_size=16):
    if freq <= 0: return None
    T = 1.0 / freq
    sigma_x = 0.4 * T
    sigma_y = 0.4 * T

    half = block_size // 2
    y, x = np.mgrid[-half:half, -half:half]

    theta_perp = theta + np.pi/2
    x_p = x*np.cos(theta_perp) + y*np.sin(theta_perp)
    y_p = -x*np.sin(theta_perp) + y*np.cos(theta_perp)

    g = np.exp(-0.5 * ((x_p**2)/(sigma_x**2 + 1e-8) +
                       (y_p**2)/(sigma_y**2 + 1e-8)))
    return g * np.cos(2*np.pi*freq*x_p)

def gabor_filtering(N, O, R, F, block_size=16, rel_thresh=0.5):
    rows, cols = N.shape
    out = np.zeros_like(N, dtype=np.float32)

    for y in range(0, rows - block_size + 1, block_size):
        for x in range(0, cols - block_size + 1, block_size):

            if R[y:y+block_size, x:x+block_size].mean() < rel_thresh:
                out[y:y+block_size, x:x+block_size] = N[y:y+block_size, x:x+block_size]
                continue

            block_f = F[y:y+block_size, x:x+block_size].mean()
            if block_f <= 0:
                out[y:y+block_size, x:x+block_size] = N[y:y+block_size, x:x+block_size]
                continue

            theta = O[y + block_size//2, x + block_size//2]
            kern = gabor_kernel(block_f, theta, block_size)

            if kern is None:
                out[y:y+block_size, x:x+block_size] = N[y:y+block_size, x:x+block_size]
                continue

            patch = N[y:y+block_size, x:x+block_size]
            filtered = cv2.filter2D(patch, -1, kern,
                                    borderType=cv2.BORDER_REFLECT_101)

            m = patch.mean()
            alpha = 0.55
            out[y:y+block_size, x:x+block_size] = alpha * np.abs(filtered - filtered.mean()) + m

    out -= out.min()
    out /= (out.max() + 1e-8)
    out_uint8 = (255*out).astype(np.uint8)

    out_clahe = cv2.createCLAHE(clipLimit=1.2, tileGridSize=(10,10)).apply(out_uint8)
    out_soft = (255 * ((out_clahe/255)**1.3)).astype(np.uint8)

    return out_uint8, out_soft

# ============================================================
#  BATCH PROCESS ALL RGBA IMAGES
# ============================================================

files = sorted(INPUT_DIR.glob("*.png"))
print(f"Found {len(files)} RGBA images.")

for f in files:
    print("Processing:", f.name)

    # 1. Load grayscale from RGBA
    img = cv2.imread(str(f), cv2.IMREAD_UNCHANGED)
    gray = img[..., 0].astype(np.float32)  # RGB are same grayscale

    # 2. Normalize
    N = normalize_fingerprint(gray)

    # 3. Orientation + reliability
    O, R = estimate_orientation_and_reliability(N, BLOCK_SIZE)

    # 4. Frequency
    F = estimate_ridge_frequency(N, O, R,
                                 BLOCK_SIZE,
                                 REL_THRESH,
                                 MIN_WAVELENGTH,
                                 MAX_WAVELENGTH)

    # 5. Gabor
    Gabor_raw, Gabor_soft = gabor_filtering(N, O, R, F,
                                            BLOCK_SIZE,
                                            REL_THRESH)

    # === SAVE OUTPUTS ===
    base = f.stem

    cv2.imwrite(str(OUTPUT_DIR/"normalized"/f"{base}_norm.png"),
                ((N-N.min())/(N.max()-N.min()+1e-8)*255).astype(np.uint8))

    cv2.imwrite(str(OUTPUT_DIR/"orientation"/f"{base}_orient.png"),
                (255*((O+np.pi/2)/np.pi)).astype(np.uint8))

    if np.any(F > 0):
        freq_norm = (255*F/np.max(F[F>0])).astype(np.uint8)
    else:
        freq_norm = F.astype(np.uint8)

    cv2.imwrite(str(OUTPUT_DIR/"frequency"/f"{base}_freq.png"), freq_norm)
    cv2.imwrite(str(OUTPUT_DIR/"gabor_raw"/f"{base}_gabor_raw.png"), Gabor_raw)
    cv2.imwrite(str(OUTPUT_DIR/"gabor_soft"/f"{base}_gabor_soft.png"), Gabor_soft)

print("DONE! All images processed.")
